# Fraser pilot: Priya LoRA fine-tune on Kaggle

This notebook clones [Priya22/safe-finetuning](https://github.com/Priya22/safe-finetuning) and runs `LoRA/run_lora.py` with the `phase1_repl` settings, reduced so a Kaggle T4 can finish one epoch.

Before running:

1. In Kaggle, set the accelerator to **GPU T4** and turn **Internet** on.
2. Open [meta-llama/Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) and accept the license with the same Hugging Face account as your token.
3. Paste the token into the next cell.
4. Run every cell from the top.

The training script is Priya's. These settings are the only changes, because the original config points at a Google Drive folder and uses a sequence length and batch size that do not fit a T4:

- data: `ft_datasets/alpaca_clean_sample_5k` from the cloned repo
- `float16` instead of `bfloat16` (a T4 cannot use bfloat16)
- sequence length 512, batch size 1, gradient accumulation 8
- 1 epoch, learning rate 2e-5, LoRA rank 64


In [ ]:
# Paste your Hugging Face token between the quotes, then run this cell.
HF_TOKEN = ""

import os
if not HF_TOKEN.strip():
    raise ValueError("Paste your Hugging Face token into HF_TOKEN, then run this cell again.")
os.environ["HF_TOKEN"] = HF_TOKEN.strip()
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()
print("Token stored for this session. It is not printed.")


In [ ]:
%%bash
set -euo pipefail
pip install -q \
  "transformers==4.51.3" \
  "trl==0.17.0" \
  "peft==0.15.2" \
  "accelerate==1.6.0" \
  "datasets==3.5.0" \
  python-box \
  sentencepiece \
  protobuf
python -c "import transformers, trl, peft; print('transformers', transformers.__version__); print('trl', trl.__version__); print('peft', peft.__version__)"


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d safe-finetuning/.git ]; then
  git clone --depth 1 https://github.com/Priya22/safe-finetuning.git
else
  echo "Repo already cloned."
fi
# Priya's script passes dtype=. transformers 4.51, which matches trl 0.17, expects torch_dtype.
sed -i 's/dtype=dtype_map\[config.dtype\]/torch_dtype=dtype_map[config.dtype]/' /kaggle/working/safe-finetuning/LoRA/run_lora.py
grep -n "torch_dtype=dtype_map" /kaggle/working/safe-finetuning/LoRA/run_lora.py
ls /kaggle/working/safe-finetuning/ft_datasets/alpaca_clean_sample_5k


In [ ]:
from huggingface_hub import login
import os

login(os.environ["HF_TOKEN"])

config_text = """
exp_id: "phase1_kaggle"
model_id: "llama1b"
dataset_id: "alpaca_clean_sample_5k"
random_state: 5
device: "cuda"
base_model_path: "meta-llama/Llama-3.2-1B-Instruct"
dtype: "float16"
max_seq_length: 512
attn_implementation: "eager"
use_liger_kernel: false
use_packing: false
target_modules:
  - "q_proj"
  - "k_proj"
  - "v_proj"
  - "o_proj"
  - "gate_proj"
  - "up_proj"
  - "down_proj"
lora_alpha: 64
lora_dropout: 0.0
lora_rank: 64
use_gradient_checkpointing: true
data_path: "/kaggle/working/safe-finetuning/ft_datasets/alpaca_clean_sample_5k"
data_template: "alpaca"
per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 8
max_grad_norm: 1.0
warmup_ratio: 0.05
lr_scheduler_type: "linear"
num_train_epochs: 1
learning_rate: 0.00002
save_strategy: "epoch"
eval_strategy: "epoch"
save_only_model: true
optim: "adamw_torch"
weight_decay: 0.01
logging_strategy: "steps"
logging_steps: 10
"""

config_path = "/kaggle/working/kaggle_phase1.yaml"
with open(config_path, "w") as f:
    f.write(config_text.strip() + "\n")
print("Wrote", config_path)


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/safe-finetuning
# Priya's script skips training if this folder already contains a tokenizer.
# Delete /kaggle/working/outputs/phase1_kaggle before a second run.
python LoRA/run_lora.py \
  --config_path /kaggle/working/kaggle_phase1.yaml \
  --save_folder /kaggle/working/outputs/phase1_kaggle


In [ ]:
%%bash
set -euo pipefail
find /kaggle/working/outputs/phase1_kaggle -maxdepth 2 -type d -o -name "adapter_config.json" -o -name "tokenizer_config.json" | head -50


When the last cell lists `adapter_config.json`, the one-epoch checkpoint is saved in `/kaggle/working/outputs/phase1_kaggle`.

Download that folder from Kaggle before the session ends. The next step, after this run finishes, is to score the base model and this checkpoint on a small slice of SORRY-Bench.
